# Core Pipeline

This notebook is the thin driver for the English pipeline.

What you can run here:
- The setup cell imports the core runner, builds `PipelineConfig`, exposes `config.stage_options["figma_redownload"]` so you can control whether cached Figma files are reused or fetched again, and exposes `config.stage_options["image_bank_seed"]` so Learning Objective / Key Takeaway image assignment stays deterministic.
- `list_core_stages()` prints the available core stages and is safe to rerun at any time.
- Each stage cell below runs one orchestrated step and prints a structured summary.

Recommended order:
1. Validate environment.
2. Generate `module_structure.json`.
3. Run Figma extraction/export.
4. Apply English post-processing.
5. Run sanitization and audits.



In [ ]:
from sea_pipeline.shared.config import PipelineConfig
from sea_pipeline.shared.logging import print_stage_result
from sea_pipeline.core.runner import list_core_stages, run_core_stage

config = PipelineConfig.from_repo_root()
config.stage_options["figma_redownload"] = False
config.stage_options["image_bank_seed"] = 20260504

list_core_stages()



## Validate Environment

Run this first.

This checks that the core pipeline can resolve its repo paths and that the required environment variables and services for the English pipeline are available before any outputs are regenerated.



In [ ]:
result = run_core_stage(config, "validate_env")
print_stage_result(result)

## Structure Generation

This regenerates the English `module_structure.json` from the table of contents and chapter metadata.

Run this when the course structure, lesson ordering, chapter metadata, or module metadata has changed.



In [ ]:
result = run_core_stage(config, "structure_generation")
print_stage_result(result)

## Figma Extraction And Export

This is the main English content build step.

It loads the cached or freshly downloaded Figma documents, extracts lesson frames, applies the core export logic, writes lesson JSON files, inserts the CSV-driven polls used by the English pipeline, performs image-related post-processing, builds the uploaded academy image bank, deterministically assigns bank photos to Learning Objective and Key Takeaway cards using `config.stage_options["image_bank_seed"]`, and keeps the detailed notebook logging from the legacy flow.



In [ ]:
result = run_core_stage(config, "figma_extraction_export")
print_stage_result(result)

## English Post-Processing

These cells run post-export enrichment steps on the English lesson outputs.

What each runnable cell does:
- `ai_avatar_tagging`: inserts AI tags and avatar-related metadata into the lesson JSONs.
- `export_lesson_ids`: exports lesson/module identifiers used by downstream steps and operational checks.



In [ ]:
result = run_core_stage(config, "ai_avatar_tagging")
print_stage_result(result)

In [ ]:
result = run_core_stage(config, "export_lesson_ids")
print_stage_result(result)

## Validation And Audits

These cells validate and audit the generated English outputs.

What each runnable cell does:
- `sanitize_json_outputs`: cleans invalid numeric values and applies JSON-level safety fixes without changing the output schema.
- `overflow_audit`: scans the generated content for likely layout overflow issues and reports anything that needs review.



In [ ]:
result = run_core_stage(config, "sanitize_json_outputs")
print_stage_result(result)

In [ ]:
result = run_core_stage(config, "overflow_audit")
print_stage_result(result)